In [1]:
import os 
import sys


In [2]:
pwd

'c:\\Users\\rajum\\DATA SCIENCE\\DeepLearning\\ImageClassification\\chiken-disease-classification-project\\research'

In [3]:
os.chdir('../')

In [17]:

import tensorflow as tf

In [18]:

model = tf.keras.models.load_model("artifacts/training/model.h5")

[2025-07-14 16:55:41,708: WARNING: saving_utils: Compiled the loaded model, but the compiled metrics have yet to be built. `model.compile_metrics` will be empty until you train or evaluate the model.]


In [26]:
#Entity Configuraton 
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class EvaluationConfig:
    path_of_model: Path
    training_data : Path
    all_params : dict
    params_image_size : list
    params_batch_size : int
    


In [27]:
os.chdir('c:\\Users\\rajum\\DATA SCIENCE\\DeepLearning\\ImageClassification\\chiken-disease-classification-project')
sys.path.append(os.path.join(os.getcwd(), "src"))


In [28]:
from cnnClassifier.constants import *
from cnnClassifier.utils .common import read_yaml, create_directories,save_json


In [29]:
class ConfigurationManager:
    def __init__(
        self, 
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])

    
    def get_validation_config(self) -> EvaluationConfig:
        eval_config = EvaluationConfig(
            path_of_model="artifacts/training/model.h5",
            training_data="artifacts/data_ingestion/Chicken-fecal-images",
            all_params=self.params,
            params_image_size=self.params.IMAGE_SIZE,
            params_batch_size=self.params.BATCH_SIZE
        )
        return eval_config

In [30]:

from urllib.parse import urlparse

In [31]:
class Evaluation:
    def __init__(self, config: EvaluationConfig):
        self.config = config

    
    def _valid_generator(self):

        datagenerator_kwargs = dict(
            rescale = 1./255,
            validation_split=0.30
        )

        dataflow_kwargs = dict(
            target_size=self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            interpolation="bilinear"
        )

        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs
        )

        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="validation",
            shuffle=False,
            **dataflow_kwargs
        )

    
    @staticmethod
    def load_model(path: Path) -> tf.keras.Model:
        return tf.keras.models.load_model(path)
    

    def evaluation(self):
        self.model = self.load_model(self.config.path_of_model)
        self._valid_generator()
        self.score = model.evaluate(self.valid_generator)

    
    def save_score(self):
        scores = {"loss": self.score[0], "accuracy": self.score[1]}
        save_json(path=Path("scores.json"), data=scores)


In [32]:
try:
    config = ConfigurationManager()
    val_config = config.get_validation_config()
    evaluation = Evaluation(val_config)
    evaluation.evaluation()
    evaluation.save_score()

except Exception as e:
   raise e

[2025-07-14 17:00:20,193: INFO: common: YAML file : config\config.yaml loaded successfully]
[2025-07-14 17:00:20,194: INFO: common: YAML file : params.yaml loaded successfully]
[2025-07-14 17:00:20,195: INFO: common: Created directory at : artifacts]
[2025-07-14 17:00:21,533: WARNING: saving_utils: Compiled the loaded model, but the compiled metrics have yet to be built. `model.compile_metrics` will be empty until you train or evaluate the model.]
Found 116 images belonging to 2 classes.


c:\Users\rajum\.conda\envs\chiken\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


8/8 ━━━━━━━━━━━━━━━━━━━━ 50s 5s/step - accuracy: 0.7504 - loss: 0.5245
[2025-07-14 17:01:12,175: INFO: common: file saved at :scores.json]
